# LTX-Video Image-to-Video (Gradio) — Free Colab T4

One-click **Google Colab** demo: [Lightricks LTX-Video](https://huggingface.co/Lightricks/LTX-Video) **image → video** with a **Gradio** UI and temporary public `*.gradio.live` link.

## Why this model

Best practical **quality / speed** on free Colab **T4 (~15 GB)** among open I2V stacks that work with **Diffusers + Gradio** (Wan 14B needs GGUF/Comfy and is much slower; SVD is weaker). Uses the **distilled** checkpoint for few-step generation.

## Defaults (T4-friendly)

| Setting | Default | Notes |
|---------|---------|--------|
| Resolution | **512 × 704** | Keep multiples of 32; lower = less VRAM / faster |
| Frames | **65** | Prefer 25, 33, 49, 65, 81 — LTX likes `(num_frames - 1)` divisible by 8 |
| Steps | **8** | Distilled; guidance_scale **1.0** |
| Expect | **several minutes** per clip | First run also downloads the model |

## Runtime

1. **Runtime → Change runtime type → GPU (T4)** → Save  
2. **Runtime → Run all**  
3. Wait for installs + model load  
4. Copy `Running on public URL: https://xxxx.gradio.live` from the last cell  

**Do not** load a spatial upsampler on free T4 (saves VRAM). Uses `float16`, `enable_model_cpu_offload()`, and VAE tiling.


## 1. GPU check


In [1]:
# Must show a CUDA device. If not: Runtime → Change runtime type → GPU → Save, then re-run.
import sys
import torch

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free/total: {free / 1024**3:.2f} / {total / 1024**3:.2f} GB")
else:
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU (T4), then re-run from the top.")


Python: 3.13.16
torch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
VRAM free/total: 14.46 / 14.56 GB


## 2. Install dependencies


In [2]:
# Torch is already on Colab. Upgrade Diffusers (needs LTXConditionPipeline) + Gradio stack.
%pip install -q -U pip
%pip install -q -U "diffusers>=0.33.0" transformers accelerate peft sentencepiece safetensors imageio imageio-ffmpeg opencv-python-headless gradio

import torch, transformers, diffusers, gradio
print("torch:", torch.__version__)
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("gradio:", gradio.__version__)


Disabling torchao because diffusers requires `torchao>=0.15.0` but found version 0.10.0. Upgrade it with `pip install -U 'torchao>=0.15.0'`.


torch: 2.14.1+cu130
diffusers: 0.43.0
transformers: 5.24.0
gradio: 6.32.0


## 3. Load pipeline

Tries **LTX-Video-0.9.7-distilled** + `LTXConditionPipeline` first (few-step I2V).  
On import/load failure or OOM, falls back to `LTXImageToVideoPipeline` + `Lightricks/LTX-Video`.


In [ ]:
import gc
import torch

DTYPE = torch.float16  # T4 has weak bf16
DISTILLED_ID = "Lightricks/LTX-Video-0.9.7-distilled"
BASE_ID = "Lightricks/LTX-Video"

USE_DISTILLED = False
pipe = None
LOAD_ERROR = None

# Clear leftover CUDA cache before load
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    from diffusers import LTXConditionPipeline
    from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition

    print(f"Loading distilled I2V: {DISTILLED_ID} ...")
    pipe = LTXConditionPipeline.from_pretrained(
        DISTILLED_ID,
        dtype=DTYPE,
    )
    # Do NOT load spatial upsampler on free T4
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    USE_DISTILLED = True
    print("Loaded LTXConditionPipeline (distilled). guidance_scale=1.0, ~8 steps.")
except Exception as e:
    LOAD_ERROR = e
    print(f"Distilled load failed ({type(e).__name__}: {e})")
    print("Falling back to LTXImageToVideoPipeline + base LTX-Video ...")
    pipe = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        from diffusers import LTXImageToVideoPipeline

        pipe = LTXImageToVideoPipeline.from_pretrained(
            BASE_ID,
            dtype=DTYPE,
        )
        pipe.enable_model_cpu_offload()
        pipe.vae.enable_tiling()
        USE_DISTILLED = False
        print("Loaded LTXImageToVideoPipeline (base). Use ~30 steps, guidance_scale=3.0.")
    except Exception as e2:
        raise RuntimeError(
            f"Could not load distilled ({LOAD_ERROR}) nor base ({e2}). "
            "Upgrade diffusers or free more disk/RAM and retry."
        ) from e2

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"After load — VRAM free/total: {free / 1024**3:.2f} / {total / 1024**3:.2f} GB")

Loading distilled I2V: Lightricks/LTX-Video-0.9.7-distilled ...


model_index.json:   0%|          | 0.00/421 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

## 4. Gradio UI + public share link

Keep this cell **running**. Copy the `Running on public URL: https://xxxx.gradio.live` line — temporary (~72h), gone when the runtime disconnects.


In [ ]:
import tempfile
import traceback

import gradio as gr
import torch
from PIL import Image
from diffusers.utils import export_to_video

DEFAULT_NEG = "worst quality, inconsistent motion, blurry, jittery, distorted"
DEFAULT_STEPS = 8 if USE_DISTILLED else 30
DEFAULT_GUIDANCE = 1.0 if USE_DISTILLED else 3.0

# Import condition helper only when distilled path is active
if USE_DISTILLED:
    from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition


def generate(image, prompt, negative_prompt, height, width, num_frames, steps, seed, guidance_scale):
    if image is None:
        raise gr.Error("Please upload an image.")
    if not (prompt or "").strip():
        raise gr.Error("Please enter a prompt.")

    height = int(height)
    width = int(width)
    num_frames = int(num_frames)
    steps = int(steps)
    seed = int(seed)
    guidance_scale = float(guidance_scale)

    if height % 32 != 0 or width % 32 != 0:
        raise gr.Error("Height and width must be multiples of 32.")

    try:
        image = image.convert("RGB").resize((width, height), Image.Resampling.LANCZOS)
        generator = torch.Generator(device="cpu").manual_seed(seed)

        if USE_DISTILLED:
            condition = LTXVideoCondition(image=image, frame_index=0)
            result = pipe(
                conditions=[condition],
                prompt=prompt.strip(),
                negative_prompt=negative_prompt or "",
                height=height,
                width=width,
                num_frames=num_frames,
                num_inference_steps=steps,
                guidance_scale=guidance_scale,
                decode_timestep=0.05,
                decode_noise_scale=0.025,
                image_cond_noise_scale=0.025,
                generator=generator,
            )
        else:
            result = pipe(
                image=image,
                prompt=prompt.strip(),
                negative_prompt=negative_prompt or "",
                height=height,
                width=width,
                num_frames=num_frames,
                num_inference_steps=steps,
                guidance_scale=guidance_scale,
                generator=generator,
            )

        frames = result.frames[0]
        out = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
        export_to_video(frames, out, fps=24)
        return out
    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise gr.Error(
            "CUDA OOM. Try 512×512, fewer frames (e.g. 33 or 49), or restart the runtime and re-run."
        )
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


with gr.Blocks(title="LTX-Video Image-to-Video") as demo:
    gr.Markdown(
        f"""# LTX-Video Image → Video
**Mode:** `{'distilled LTXConditionPipeline (few-step)' if USE_DISTILLED else 'base LTXImageToVideoPipeline'}`
T4 tips: start at **512×704**, **65** frames, **{DEFAULT_STEPS}** steps. Expect several minutes per clip.
"""
    )
    with gr.Row():
        with gr.Column():
            image = gr.Image(type="pil", label="Input image (required)")
            prompt = gr.Textbox(
                label="Prompt (required)",
                lines=3,
                placeholder="A cinematic shot of ...",
            )
            negative_prompt = gr.Textbox(
                label="Negative prompt",
                value=DEFAULT_NEG,
                lines=2,
            )
            with gr.Row():
                height = gr.Slider(256, 768, value=512, step=32, label="Height")
                width = gr.Slider(256, 768, value=704, step=32, label="Width")
            num_frames = gr.Dropdown(
                choices=[25, 33, 49, 65, 81],
                value=65,
                label="Num frames (odd / LTX-compatible)",
            )
            with gr.Row():
                steps = gr.Slider(1, 50, value=DEFAULT_STEPS, step=1, label="Inference steps")
                guidance_scale = gr.Slider(
                    0.0, 10.0, value=DEFAULT_GUIDANCE, step=0.1, label="Guidance scale"
                )
            seed = gr.Number(value=42, precision=0, label="Seed")
            btn = gr.Button("Generate video", variant="primary")
        with gr.Column():
            video = gr.Video(label="Output")

    btn.click(
        fn=generate,
        inputs=[image, prompt, negative_prompt, height, width, num_frames, steps, seed, guidance_scale],
        outputs=video,
    )

demo.queue(max_size=2)
demo.launch(share=True, debug=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/AI_Models"
os.makedirs(MODEL_DIR, exist_ok=True)